# S4 — Wav2Vec2-Large-960h SER
## AudioGuard FYP | SER Track
**Purpose**: Fine-tune `facebook/wav2vec2-large-960h` using the HuggingFace `Trainer` API. This notebook demonstrates a more streamlined setup compared to the custom loops in S2/S3. The feature extractor is frozen to preserve low-level speech representations.

**Expected runtime**: ~20–40 min on GPU / Not recommended on CPU

**Output**: `./outputs/S4_wav2vec2_large_ser/`

In [ ]:
%pip install transformers datasets accelerate librosa soundfile scikit-learn pandas numpy tqdm torch torchvision torchaudio matplotlib seaborn --quiet
print('✓ Dependencies installed')

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import librosa
import torch
from pathlib import Path
from transformers import (
    Wav2Vec2ForSequenceClassification,
    AutoFeatureExtractor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
 
warnings.filterwarnings('ignore')
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
CONFIG = {
    "model_id"      : "S4",
    "model_name"    : "facebook/wav2vec2-large-960h",
    "num_labels"    : 7,
    "epochs"        : 15,          # was 5 — needs more epochs
    "batch_size"    : 8,
    "grad_accum"    : 4,           # was 2 — effective batch = 32
    "learning_rate" : 5e-5,        # was 2e-5 — counterintuitively go higher but with proper clipping
    "warmup_ratio"  : 0.1,         # 10% of total steps — replaces fixed warmup_steps
    "max_audio_len" : 16000 * 6,   # was 10s — trim to 6s, speeds up + reduces noise
    "test_size"     : 0.15,
    "val_size"      : 0.15,
    "seed"          : 42,
    "output_dir"    : "/kaggle/working/",
    "ravdess_root"  : "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio",
    "tess_root"     : "/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess",
    "device"        : "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"Output dir : {CONFIG['output_dir']}")
print(f"Device     : {CONFIG['device']}")

In [ ]:
# RAVDESS encodes emotion as the 3rd dash-separated token in the filename
RAVDESS_CODE = {
    '01': 'neutral', '02': 'calm',  '03': 'happy',    '04': 'sad',
    '05': 'angry',   '06': 'fearful','07': 'disgust',  '08': 'surprised',
}
 
# Unified 7-class set  ('calm' dropped — merges with neutral in literature)
LABEL2ID = {
    'neutral': 0, 'happy': 1, 'sad': 2, 'angry': 3,
    'fearful': 4, 'disgust': 5, 'surprised': 6,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
 
# TESS folder suffix → unified label
TESS_MAP = {
    'neutral': 'neutral', 'happy': 'happy', 'sad': 'sad',
    'angry'  : 'angry',   'fear' : 'fearful', 'disgust': 'disgust',
    'ps'     : 'surprised',   # "pleasant surprise"
}
print(f"Labels ({CONFIG['num_labels']}): {list(LABEL2ID.keys())}")

In [ ]:
records = []
 
# RAVDESS
ravdess_files = glob.glob(os.path.join(CONFIG["ravdess_root"], "**", "*.wav"), recursive=True)
print(f"RAVDESS files : {len(ravdess_files)}")
for fp in ravdess_files:
    parts = Path(fp).stem.split('-')
    if len(parts) < 3:
        continue
    emotion = RAVDESS_CODE.get(parts[2])
    if emotion is None or emotion == 'calm' or emotion not in LABEL2ID:
        continue
    records.append({'path': fp, 'label': LABEL2ID[emotion], 'emotion': emotion, 'source': 'RAVDESS'})
 
# TESS
tess_files = glob.glob(os.path.join(CONFIG["tess_root"], "**", "*.wav"), recursive=True)
print(f"TESS files    : {len(tess_files)}")
for fp in tess_files:
    emo_raw = Path(fp).parent.name.lower().split('_')[-1]   # e.g. OAF_angry → angry
    emotion = TESS_MAP.get(emo_raw)
    if emotion is None:
        continue
    records.append({'path': fp, 'label': LABEL2ID[emotion], 'emotion': emotion, 'source': 'TESS'})
 
df = pd.DataFrame(records)
print(f"\nTotal samples : {len(df)}")
print(df.groupby(['source', 'emotion']).size().to_string())

In [ ]:
train_val_df, test_df = train_test_split(
    df, test_size=CONFIG["test_size"],
    stratify=df["label"], random_state=CONFIG["seed"]
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=CONFIG["val_size"] / (1 - CONFIG["test_size"]),
    stratify=train_val_df["label"], random_state=CONFIG["seed"]
)
print(f"\nSplit → train: {len(train_df)}  |  val: {len(val_df)}  |  test: {len(test_df)}")

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(CONFIG["model_name"])
SAMPLING_RATE = feature_extractor.sampling_rate   # 16000
 
def load_audio(path: str) -> np.ndarray:
    waveform, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
    return waveform
 
def build_hf_dataset(split_df: pd.DataFrame) -> Dataset:
    audio_arrays = [load_audio(p) for p in split_df["path"]]
    encodings = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLING_RATE,
        max_length=CONFIG["max_audio_len"],
        truncation=True,
        padding=True,
        return_tensors="np",
    )
    ds = Dataset.from_dict({
        "input_values": list(encodings["input_values"]),
        "labels"       : split_df["label"].tolist(),
    })
    ds.set_format("torch")
    return ds
 
print("Building datasets (this takes a few minutes) …")
train_dataset = build_hf_dataset(train_df)
val_dataset   = build_hf_dataset(val_df)
test_dataset  = build_hf_dataset(test_df)
print("Datasets ready ✓")

In [ ]:
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["num_labels"],
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

# Freeze the CNN feature encoder manually (replaces deprecated freeze_feature_extractor())
# Freeze CNN feature extractor
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

# Also freeze bottom 12 transformer layers (of 24)
for i, layer in enumerate(model.wav2vec2.encoder.layers):
    if i < 12:
        for param in layer.parameters():
            param.requires_grad = False

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)")
# Expect ~50% trainable now

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1"      : f1_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir                  = CONFIG["output_dir"],
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = CONFIG["learning_rate"],
    per_device_train_batch_size = CONFIG["batch_size"],
    per_device_eval_batch_size  = CONFIG["batch_size"],
    gradient_accumulation_steps = CONFIG["grad_accum"],
    num_train_epochs            = CONFIG["epochs"],
    warmup_ratio                = CONFIG["warmup_ratio"],  # replaces warmup_steps
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,     # ← gradient clipping, was missing
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 20,
    save_total_limit            = 2,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
    seed                        = CONFIG["seed"],
)

In [ ]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
 

In [ ]:
print("\nStarting training …")
train_result = trainer.train()
print("Training complete ✓")
print(train_result.metrics)

In [ ]:
trainer.save_model(CONFIG["output_dir"])
feature_extractor.save_pretrained(CONFIG["output_dir"])
print(f"Model saved to {CONFIG['output_dir']}")

In [ ]:
test_results = trainer.predict(test_dataset)
preds  = np.argmax(test_results.predictions, axis=-1)
labels = test_results.label_ids
 
print("\n── Test Set Results ──────────────────────────────")
print(f"  Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"  Macro F1 : {f1_score(labels, preds, average='macro', zero_division=0):.4f}")
print("\nPer-class report:")
print(classification_report(labels, preds, target_names=list(LABEL2ID.keys()), zero_division=0))

In [ ]:
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(LABEL2ID.keys()),
            yticklabels=list(LABEL2ID.keys()), ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('S4 — Confusion Matrix (Test Set)')
plt.tight_layout()
cm_path = os.path.join(CONFIG["output_dir"], "confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved to {cm_path}")

In [ ]:
metrics_out = {
    "model_id"     : CONFIG["model_id"],
    "model_name"   : CONFIG["model_name"],
    "test_accuracy": float(accuracy_score(labels, preds)),
    "test_macro_f1": float(f1_score(labels, preds, average='macro', zero_division=0)),
    "train_metrics": train_result.metrics,
}
metrics_path = os.path.join(CONFIG["output_dir"], "metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved → {metrics_path}")
print(json.dumps(metrics_out, indent=2))